# 04 — Start and monitor a pipeline

Use the same operations as Studio from Python: **start, check status, find
outputs, wait, stop**. These are the API calls an external scheduler can use.

Run notebook 02 first. Choose a model and training or inference below;
the pipeline already has its data input and other defaults. Run All starts
one preliminary training execution for the first deployed model. Waiting and
stopping are optional and off by default.

To run inference, first approve the intended model in this namespace using
Studio or notebook 02. Set `PIPELINE_KIND = "inference"` and rerun Choose
and Start. Inference uses the latest eligible approved model by default.
Notebook 05 covers the automated training → approval → inference test.

## Load the deployed pipeline names

- Read the namespace from the same config.json used by notebook 02.
- Load its saved training/inference names and model IDs.
- Create the SageMaker client; no job starts in this cell.

- Show complete table values and all columns so paths and ARNs are not truncated.


In [ ]:
from pathlib import Path
import json
import time
import boto3
import pandas as pd
from botocore.config import Config
from IPython.display import display

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "mlops-models/src/mlops_models").is_dir())
config = json.loads((ROOT / "notebooks/config.json").read_text())
namespace = config["namespace"]
publication_path = ROOT / ".runs/source-deploy" / namespace / "publication.json"
if not namespace or not publication_path.is_file():
    raise ValueError("Run notebook 02 for this namespace first.")
publication = json.loads(publication_path.read_text())
session = boto3.Session(region_name=publication["region"])
sm = session.client(
    "sagemaker", config=Config(connect_timeout=10, read_timeout=30,
                              retries={"mode": "standard", "total_max_attempts": 3}))
print("Namespace:", namespace, "| Models:", publication["models"])
print("CSV saved at deployment:", publication.get("csv_input_uri") or "Feature Store input")


## Choose what to run

- Set PIPELINE_KIND to training or inference, just as you select a pipeline in Studio.
- Keep the first deployed model, or set MODEL_ID to a name printed above.
- The Start cell below shows the parameter choices; the deployed pipeline supplies the CSV input.

In [ ]:
PIPELINE_KIND = "training"  # "training" or "inference"
MODEL_ID = publication["models"][0]
pipeline_name = publication["pipelines"][PIPELINE_KIND]["name"]
print("Selected:", pipeline_name, "| Model:", MODEL_ID)


## Start the selected pipeline

- Run start_pipeline_execution with the selected pipeline name and ModelId.
- Leave the other entries commented to use deployed defaults, or uncomment the choices for your pipeline.
- Keep the returned execution ARN for status, output, wait and stop calls.

**Each run of this cell starts a new execution.** Preliminary training creates
metadata and ignores metadata input. Final training requires the S3 URI of the
preliminary run's `evaluation/model_metadata.json`.

Inference defaults to `prediction_inference`. Use `prediction_test` for labelled
test data. `ModelCodeSource="pipeline"` uses the deployed inference code;
`"artifact"` uses code saved with the model. A reference execution is optional
for comparison; leave it out for the ordinary prediction path.

In [ ]:
response = sm.start_pipeline_execution(
    PipelineName=pipeline_name,
    PipelineParameters=[
        {"Name": "ModelId", "Value": MODEL_ID},
        # Training: uncomment BOTH entries below for final training.
        # {"Name": "TrainingMode", "Value": "Final model with all data"},
        # {"Name": "ModelMetadataUri", "Value": "s3://bucket/path/evaluation/model_metadata.json"},
        # Inference: uncomment only the choices you want to change.
        # {"Name": "PredictionAnalysis", "Value": "prediction_test"},
        # {"Name": "ModelCodeSource", "Value": "pipeline"},
        # {"Name": "ReferenceExecutionArn", "Value": "arn:aws:sagemaker:region:account:pipeline/name/execution/id"},
    ],
)
execution_arn = response["PipelineExecutionArn"]
print("Execution:", execution_arn)

## Check status and outcome

- Read the execution started above, or paste an existing execution ARN.
- Show the overall status and each step's status and failure reason.
- Rerun this cell to refresh; it does not start another execution.

For an existing execution, run Setup and this cell only. Use an ARN from
the same AWS region as the loaded deployment.


In [ ]:
MONITOR_EXECUTION_ARN = ""  # Optional: paste an existing execution ARN.
selected_execution = MONITOR_EXECUTION_ARN.strip() or globals().get("execution_arn", "")
if not selected_execution:
    raise ValueError("Start an execution above, or paste its ARN here.")
execution = sm.describe_pipeline_execution(PipelineExecutionArn=selected_execution)
print("Execution:", selected_execution)
print("Status:", execution["PipelineExecutionStatus"])
if execution.get("FailureReason"):
    print("Failure:", execution["FailureReason"])
steps = []
for page in sm.get_paginator("list_pipeline_execution_steps").paginate(
        PipelineExecutionArn=selected_execution):
    steps.extend(page["PipelineExecutionSteps"])
display(pd.DataFrame([
    {"step": s["StepName"], "status": s["StepStatus"], "failure": s.get("FailureReason", "")}
    for s in steps], columns=["step", "status", "failure"]))


## Optional: wait for completion

- Change WAIT_FOR_COMPLETION to True to poll the selected execution every 30 seconds.
- Stop waiting when it succeeds, fails or stops, or after about two hours.
- Print the outcome; rerun Check status for step details. Interrupting this cell leaves AWS jobs running.


In [ ]:
WAIT_FOR_COMPLETION = False
if WAIT_FOR_COMPLETION:
    for attempt in range(241):
        execution = sm.describe_pipeline_execution(PipelineExecutionArn=selected_execution)
        status = execution["PipelineExecutionStatus"]
        print(status, execution.get("FailureReason", ""), flush=True)
        if status in {"Succeeded", "Failed", "Stopped"}:
            break
        if attempt == 240:
            print("Wait limit reached; the execution is still running.")
            break
        time.sleep(30)


## Find outputs from producing jobs

- Refresh the selected execution's steps, including after the optional wait above.
- Describe the jobs named by those steps to find their actual S3 output locations.
- Show the producing job or model package ARN alongside each location; rerun as more steps finish.

Cached steps may point to an earlier job's outputs. The table uses the returned
job ARNs rather than guessing a folder from the current execution ID. Empty
results mean no output locations have been reported yet.

In [ ]:
steps = []
for page in sm.get_paginator("list_pipeline_execution_steps").paginate(
        PipelineExecutionArn=selected_execution):
    steps.extend(page["PipelineExecutionSteps"])

artifact_rows = []
for step in steps:
    for kind, item in step.get("Metadata", {}).items():
        arn = item.get("Arn") if isinstance(item, dict) else None
        if not arn:
            continue
        name = arn.rsplit("/", 1)[-1]
        outputs = {}
        if kind == "ProcessingJob":
            job = sm.describe_processing_job(ProcessingJobName=name)
            outputs = {output["OutputName"]: output.get("S3Output", {}).get("S3Uri", "")
                       for output in job.get("ProcessingOutputConfig", {}).get("Outputs", [])}
        elif kind == "TrainingJob":
            job = sm.describe_training_job(TrainingJobName=name)
            outputs = {
                "model": job.get("ModelArtifacts", {}).get("S3ModelArtifacts", ""),
                "training output prefix": job.get("OutputDataConfig", {}).get("S3OutputPath", ""),
            }
        elif kind == "TransformJob":
            job = sm.describe_transform_job(TransformJobName=name)
            outputs = {"transform": job.get("TransformOutput", {}).get("S3OutputPath", "")}
        elif kind == "RegisterModel":
            outputs = {"model package": arn}
        for label, uri in outputs.items():
            if uri:
                artifact_rows.append({"step": step["StepName"], "artifact": label,
                                      "uri": uri, "producer": arn})
display(pd.DataFrame(artifact_rows, columns=["step", "artifact", "uri", "producer"]))

## Read one of the discovered reports

- Choose an existing report location from the completed jobs in the table above.
- Populate and print JSON_ARTIFACT_URI automatically; no path needs to be pasted.
- Read that JSON object from S3 and display it; rerun Find outputs first when more jobs finish.

The example reads a model metadata file or a model registration/authorization report,
using the actual producing job's output prefix. It starts no new execution.


In [ ]:
report_files = {"evaluation": "model_metadata.json",
                "registration": "package.json", "authorization": "package.json"}
completed_steps = {step["StepName"] for step in steps if step["StepStatus"] == "Succeeded"}
JSON_ARTIFACT_URI = next((
    row["uri"].rstrip("/") + "/" + report_files[row["artifact"]]
    for row in artifact_rows
    if row["step"] in completed_steps and row["artifact"] in report_files
), "")

if JSON_ARTIFACT_URI:
    print("JSON artifact:", JSON_ARTIFACT_URI)
    bucket, key = JSON_ARTIFACT_URI.removeprefix("s3://").split("/", 1)
    response = session.client("s3").get_object(Bucket=bucket, Key=key)
    with response["Body"] as stream:
        report = json.load(stream)
    display(report)
else:
    print("No completed JSON report yet. Refresh Find outputs after evaluation or model resolution finishes.")


## Optional: stop the selected execution

- Keep STOP_EXECUTION False to leave the execution running.
- Set it to True and run this cell to request a stop for selected_execution.
- Rerun Check status afterward; a stop request does not mean every job has already stopped.


In [ ]:
STOP_EXECUTION = False
if STOP_EXECUTION:
    sm.stop_pipeline_execution(PipelineExecutionArn=selected_execution)
    print("Stop requested:", selected_execution)
